# 剑鱼自动导出：每日400条
单文件包含全部代码。从2025-01-03起，自动读取原进度；启动最后一段后等待北京时间2026-09-14 03:00；之后每天03:00继续，不再确认。
必须保持电脑开机、不休眠，Chrome、Jupyter内核和网络可用。不是关机后仍运行的后台服务。
账号密码仅在本机隐藏输入，保存在内核内存，不写入文件。遇到短信、滑块、扫码或其他登录验证会停止，需要人工处理。
取消一批省份后停止；持续处理后续日期。预算够时优先全国，预算不足时读取当天实际地区、按拼音顺序拆分，按用户授权记录并省略非省份覆盖差额。400是每日上限，不保证正好凑满。省份401～800条尚未支持更细拆分，会停止提示。


## 1．安装依赖（首次运行）

In [ ]:
%pip install "selenium>=4.49,<5"


## 2．加载全部代码
直接运行，无需外部文件。

In [ ]:
import sys
import types
from pathlib import Path
MODULE_SOURCES = {'email_notice': '"""SMTP 超额通知。密码只读取环境变量，不写入任务文件。"""\nimport hashlib\nimport os\nimport smtplib\nimport ssl\nfrom email.message import EmailMessage\n\n\nclass SMTPNotifier:\n    def __init__(self, host, port, sender, recipient, username=\'\', password=\'\', security=\'ssl\'):\n        if not host or not sender or not recipient:\n            raise ValueError(\'必须填写 SMTP 主机、发件人和收件人\')\n        if security not in {\'ssl\', \'starttls\'}:\n            raise ValueError(\'SMTP_SECURITY 必须为 ssl 或 starttls\')\n        self.host, self.port = host, int(port)\n        self.sender, self.recipient = sender, recipient\n        self.username, self.password, self.security = username, password, security\n\n    @classmethod\n    def from_env(cls):\n        return cls(os.environ.get(\'SMTP_HOST\', \'\'), os.environ.get(\'SMTP_PORT\', \'465\'),\n                   os.environ.get(\'SMTP_FROM\', \'\'), os.environ.get(\'ALERT_TO\', \'zihao.zhang@smartx.com\'),\n                   os.environ.get(\'SMTP_USER\', \'\'), os.environ.get(\'SMTP_PASSWORD\', \'\'),\n                   os.environ.get(\'SMTP_SECURITY\', \'ssl\'))\n\n    def send(self, record):\n        message = EmailMessage()\n        message[\'From\'], message[\'To\'] = self.sender, self.recipient\n        message[\'Subject\'] = f"剑鱼导出超额：{record[\'date\']} {record[\'region\']} {record[\'count\']}条，已跳过"\n        key = hashlib.sha256(f"{record[\'date\']}|{record[\'region\']}|{self.recipient}".encode()).hexdigest()\n        message[\'Message-ID\'] = f\'<jianyu-{key}@export.local>\'\n        message.set_content(\n            f"标讯日期：{record[\'date\']}\\n省份：{record[\'region\']}\\n"\n            f"查询数据量：{record[\'count\']} 条\\n每日额度：800 条\\n"\n            "关键词：超融合、分布式存储、私有云、虚拟化；匹配方式：全选。\\n"\n            "处理结果：按配置放弃该日期该省份的导出，未扣除该省份额度，继续处理后续地区。\\n"\n            "该记录保存在 range_progress.json 的 skipped 中，不计入已导出数据。\\n")\n        context = ssl.create_default_context()\n        if self.security == \'ssl\':\n            client = smtplib.SMTP_SSL(self.host, self.port, timeout=30, context=context)\n        else:\n            client = smtplib.SMTP(self.host, self.port, timeout=30)\n        with client:\n            if self.security == \'starttls\':\n                client.starttls(context=context)\n            if self.username:\n                client.login(self.username, self.password)\n            refused = client.send_message(message)\n            if refused:\n                raise RuntimeError(\'收件人被邮件服务器拒绝\')\n', 'export_range': '"""按日期、地区顺序调度。后端负责页面操作；本模块不依赖 Selenium。"""\nimport json\nfrom datetime import date, datetime, timedelta, timezone\nfrom pathlib import Path\n\n# 拼音顺序；实际地区以网站返回的完整列表为准，使用此表排序。\nPROVINCES = \'安徽 澳门 北京 重庆 福建 甘肃 广东 广西 贵州 海南 河北 黑龙江 河南 湖北 湖南 江苏 江西 吉林 辽宁 内蒙古 宁夏 青海 山东 上海 山西 陕西 四川 台湾 天津 香港 新疆 西藏 云南 浙江\'.split()\n\n\ndef province_key(name):\n    matches = [i for i, prefix in enumerate(PROVINCES) if name.startswith(prefix)]\n    if len(matches) != 1:\n        raise ValueError(f\'未知地区，需明确拼音排序：{name}\')\n    return matches[0]\n\n\ndef china_today():\n    return datetime.now(timezone(timedelta(hours=8))).date()\n\n\nclass ExportRange:\n    """backend: balance(), query(day, regions), regions(day),\n    prepare(day, regions, count), submit(batch_id), recover(pending)。\n    prepare 返回 (实际当日余额, 待提交条数)，不得扣额度；\n    submit/recover 返回经过 XLSX 校验的文件路径。地区使用字符串路径。\n    notifier.send(record)：发送超额通知；失败记录到持久化发件箱。\n    """\n    def __init__(self, backend, base, start=\'2025-01-01\', end=None, confirm=False,\n                 today=china_today, notifier=None, daily_limit=200):\n        self.backend, self.today, self.confirm = backend, today, confirm\n        self.notifier = notifier\n        if type(daily_limit) is not int or not 1 <= daily_limit <= 800:\n            raise ValueError(\'daily_limit必须为1至800的整数\')\n        self.daily_limit = daily_limit\n        self.base = Path(base)\n        self.base.mkdir(parents=True, exist_ok=True)\n        self.path = self.base / \'range_progress.json\'\n        self.lock = self.base / \'one_day.lock\'  # 与旧入口共用锁，防止同时扣额度。\n        self.start = date.fromisoformat(start)\n        self.end = date.fromisoformat(end) if end else today()\n        if self.end < self.start:\n            raise ValueError(\'结束日期早于开始日期\')\n        self.state = json.loads(self.path.read_text(encoding=\'utf-8\')) if self.path.exists() else {\n            \'start\': start, \'current_date\': start, \'queue\': None, \'completed\': [],\n            \'pending\': None, \'quota_day\': None, \'remaining\': 800, \'status\': \'ready\',\n            \'skipped\': [], \'notifications\': [],\n        }\n        if self.state[\'start\'] != start:\n            raise ValueError(\'恢复任务的开始日期与原任务不同，请使用独立目录\')\n\n    def save(self):\n        temp = self.path.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(self.state, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        temp.replace(self.path)\n\n    def count(self, regions):\n        value = self.backend.query(self.state[\'current_date\'], regions)\n        if type(value) is not int or value < 0:\n            raise RuntimeError(\'查询条数必须是非负整数\')\n        return value\n\n    def next_export_day(self):\n        """先跳过已完成或零条日期，再用实际待处理日期读取余额。"""\n        while date.fromisoformat(self.state[\'current_date\']) <= self.end:\n            if self.state[\'queue\']:\n                return True\n            if self.state[\'queue\'] is None:\n                previous = (self.backend.existing_day(self.state[\'current_date\'])\n                            if hasattr(self.backend, \'existing_day\') else None)\n                if previous:\n                    self.state[\'completed\'].append({\n                        \'date\': self.state[\'current_date\'], \'regions\': [],\n                        \'file\': str(previous[\'file\']), \'count\': previous[\'count\'],\n                        \'source\': \'legacy_one_day\'})\n                elif self.count([]) > 0:\n                    return True\n            self.state.update(current_date=(date.fromisoformat(self.state[\'current_date\'])\n                                           + timedelta(days=1)).isoformat(), queue=None)\n            self.save()\n        return False\n\n    def finish_pending(self, file):\n        if not file or not Path(file).is_file():\n            raise RuntimeError(\'未取得已校验的下载文件；保留 pending，不重新提交\')\n        pending = self.state[\'pending\']\n        self.state[\'completed\'].append({**pending, \'file\': str(file)})\n        del self.state[\'queue\'][:len(pending[\'items\'])]\n        self.state[\'pending\'] = None\n        self.save()\n\n    def run(self):\n        with self.lock.open(\'x\', encoding=\'utf-8\') as handle:\n            handle.write(\'运行中；仅在确认旧进程结束后清理此锁\')\n        try:\n            self._mail_attempted = set()\n            # 在锁内重新加载，避免同一进程预先创建的第二个任务覆盖进度。\n            if self.path.exists():\n                self.state = json.loads(self.path.read_text(encoding=\'utf-8\'))\n            if self.state[\'start\'] != self.start.isoformat():\n                raise ValueError(\'进度开始日期不一致\')\n            self.state.setdefault(\'skipped\', [])\n            self.state.setdefault(\'notifications\', [])\n            return self._run()\n        finally:\n            self.lock.unlink(missing_ok=True)\n\n    def _run(self):\n        if self.confirm and self.notifier is None:\n            raise ValueError(\'正式运行前必须配置超额邮件通知\')\n        self.notify_pending()\n        if self.state[\'pending\']:\n            # 恢复只下载原订单，绝不再次 submit。\n            self.finish_pending(self.backend.recover(self.state[\'pending\']))\n        checked_day = None\n        while date.fromisoformat(self.state[\'current_date\']) <= self.end:\n            if not self.next_export_day():\n                break\n            if self.state[\'queue\'] == []:\n                next_day = date.fromisoformat(self.state[\'current_date\']) + timedelta(days=1)\n                self.state.update(current_date=next_day.isoformat(), queue=None)\n                self.save()\n                continue\n            if self.state[\'queue\'] is None and hasattr(self.backend, \'existing_day\'):\n                previous = self.backend.existing_day(self.state[\'current_date\'])\n                if previous:\n                    self.state[\'completed\'].append({\n                        \'date\': self.state[\'current_date\'], \'regions\': [],\n                        \'file\': str(previous[\'file\']), \'count\': previous[\'count\'],\n                        \'source\': \'legacy_one_day\'})\n                    self.state[\'queue\'] = []\n                    self.save()\n                    continue\n            quota_day = self.today().isoformat()\n            if checked_day != quota_day:\n                self.backend.balance_day = self.state[\'current_date\']\n                balance = self.backend.balance()\n                if type(balance) is not int or not 0 <= balance <= 800:\n                    raise RuntimeError(\'无法读取当日实际剩余额度\')\n                if self.today().isoformat() != quota_day:\n                    continue\n                used = sum(p[\'count\'] for p in self.state[\'completed\']\n                           if p.get(\'quota_day\') == quota_day)\n                allowance = max(0, self.daily_limit - used)\n                remaining = (min(balance, allowance, self.state[\'remaining\'])\n                             if self.state[\'quota_day\'] == quota_day else min(balance, allowance))\n                self.state.update(quota_day=quota_day, remaining=remaining)\n                checked_day = quota_day\n                self.save()\n            remaining = self.state[\'remaining\']\n            if not remaining:\n                return self.stop(\'waiting_quota\')\n            if self.state[\'queue\'] is None:\n                count = self.count([])\n                if count <= remaining:\n                    self.state[\'queue\'] = [{\'region\': None, \'count\': count}]\n                else:\n                    regions = self.backend.regions(self.state[\'current_date\'])\n                    if not regions or len(set(regions)) != len(regions):\n                        raise RuntimeError(\'省份清单为空或重复\')\n                    regions = sorted(regions, key=province_key)\n                    items = [{\'region\': r, \'count\': self.count([r])} for r in regions]\n                    if sum(i[\'count\'] for i in items) != count:\n                        raise RuntimeError(\'省份合计与全国不一致；检查遗漏地区或数据变动\')\n                    self.state[\'queue\'] = items\n                self.save()\n            queue = self.state[\'queue\']\n            while queue and queue[0][\'count\'] == 0:\n                queue.pop(0)\n            if not queue:\n                next_day = date.fromisoformat(self.state[\'current_date\']) + timedelta(days=1)\n                self.state.update(current_date=next_day.isoformat(), queue=None)\n                self.save()\n                continue\n            if queue[0][\'count\'] > 800:\n                if not self.confirm:\n                    return self.stop(\'preview_skip\')\n                item = queue[0]\n                if item[\'region\'] is None:\n                    raise RuntimeError(\'超额全国数据必须先按省份查询\')\n                # 跳过之前重新核对，避免根据过期条数丢弃省份。\n                fresh = self.count([item[\'region\']])\n                if fresh != item[\'count\']:\n                    raise RuntimeError(\'待跳过省份条数发生变化，请核对队列\')\n                record = {\'date\': self.state[\'current_date\'], **item,\n                          \'reason\': \'province_over_daily_limit\', \'limit\': 800}\n                self.state[\'skipped\'].append(record)\n                self.state[\'notifications\'].append({**record, \'status\': \'pending\', \'attempts\': 0})\n                queue.pop(0)\n                self.save()\n                self.notify_pending()\n                continue\n            selected, count = [], 0\n            for item in queue:\n                if count + item[\'count\'] > remaining:\n                    break  # 保持拼音顺序，不跳过省份拼凑。\n                selected.append(item)\n                count += item[\'count\']\n            if not selected:\n                # 200条自定上限不是网站800条硬限制，不能据此永久跳过省份。\n                return self.stop(\'needs_finer_split\' if queue[0][\'count\'] > self.daily_limit\n                                 else \'waiting_quota\')\n            regions = [i[\'region\'] for i in selected if i[\'region\'] is not None]\n            if self.count(regions) != count:\n                raise RuntimeError(\'合并查询条数变化或地区重叠，未提交；请核对当前队列\')\n            balance, order_count = self.backend.prepare(self.state[\'current_date\'], regions, count)\n            if type(balance) is not int or not 0 <= balance <= 800 or order_count != count:\n                raise RuntimeError(\'订单余额或条数校验失败\')\n            if self.today().isoformat() != quota_day:\n                continue  # 跨午夜重新核对额度和查询。\n            if balance < remaining:\n                self.state[\'remaining\'] = balance\n                # 全国整日不再能放入实际余额，改为省份队列。\n                if queue[0][\'region\'] is None:\n                    self.state[\'queue\'] = None\n                self.save()\n                continue\n            if not self.confirm:\n                return self.stop(\'preview\')\n            from uuid import uuid4\n            pending = {\'batch_id\': uuid4().hex, \'date\': self.state[\'current_date\'],\n                       \'regions\': regions, \'items\': selected, \'count\': count,\n                       \'quota_day\': quota_day}\n            self.state.update(pending=pending, remaining=remaining-count, status=\'submitting\')\n            self.save()  # 必须先持久化，再执行唯一一次扣额。\n            self.finish_pending(self.backend.submit(pending[\'batch_id\']))\n        return self.stop(\'complete_with_skips\' if self.state[\'skipped\'] else \'complete\')\n\n    def notify_pending(self):\n        if not self.confirm:\n            return\n        for record in self.state[\'notifications\']:\n            if record[\'status\'] == \'sent\':\n                continue\n            # 每次 run 最多重试每条一次；每次跳过新省份不会重复重试旧失败邮件。\n            key = (record[\'date\'], record[\'region\'])\n            if key in getattr(self, \'_mail_attempted\', set()):\n                continue\n            if not hasattr(self, \'_mail_attempted\'):\n                self._mail_attempted = set()\n            self._mail_attempted.add(key)\n            record[\'attempts\'] += 1\n            try:\n                if self.notifier is None:\n                    raise RuntimeError(\'未配置发信服务\')\n                self.notifier.send(record)\n                record.update(status=\'sent\', error=None)\n            except Exception as error:\n                # 不保存异常正文，SMTP 错误可能包含服务器账户信息。\n                record.update(status=\'failed\', error=type(error).__name__)\n                print(f"邮件未发送：{record[\'date\']} {record[\'region\']}；已保留待重试。")\n            self.save()\n\n    def stop(self, status):\n        self.state[\'status\'] = status\n        self.state[\'notification_failures\'] = sum(n[\'status\'] != \'sent\' for n in self.state[\'notifications\'])\n        self.state[\'end\'] = self.end.isoformat()\n        self.save()\n        return self.state\n', 'export_one_day': '"""Jupyter 单日导出；复用已连接且已登录的 Selenium driver。\n\n入口：export_one_day(driver, \'2025-01-02\')。\n使用网站已保存的四关键词配置；不臆测关键词编辑弹窗。\n"""\nimport json\nimport re\nimport time\nfrom datetime import date, datetime\nfrom decimal import Decimal\nfrom pathlib import Path\nfrom uuid import uuid4\nfrom zipfile import ZipFile, BadZipFile\n\nfrom selenium.webdriver.common.by import By\nfrom selenium.webdriver.common.keys import Keys\nfrom selenium.webdriver.support.ui import WebDriverWait\nfrom selenium.common.exceptions import (\n    NoSuchWindowException, StaleElementReferenceException, TimeoutException,\n)\n\nFILTER_URL = (\'https://www.jianyu360.cn/page_workDesktop/work-bench/page\'\n              \'?link=https%3A%2F%2Fwww.jianyu360.cn%2Ffront%2FdataExport%2FtoSieve\')\nWORDS = {\'超融合\', \'分布式存储\', \'私有云\', \'虚拟化\'}\nVERSION = \'2026-09-13-range-prefix-100-v6\'\n\n\ndef calendar_month(text):\n    year = re.search(r\'(\\d{4})\\s*年\', text)\n    month = re.search(r\'(\\d{1,2})\\s*月\', text)\n    if not year or not month or not 1 <= int(month.group(1)) <= 12:\n        raise RuntimeError(f\'无法识别日历年月：{text!r}\')\n    return int(year.group(1)), int(month.group(1))\n\n\ndef order_counts(text):\n    """只允许明确的免费条数结算。通用购买须知不作为订单金额。"""\n    for match in re.finditer(r\'实付金额|应付金额|应付总额\', text):\n        number = re.match(r\'\\s*[:：]?\\s*[¥￥]?\\s*(\\d+(?:\\.\\d+)?)\\s*(?:元)?(?=\\s|$)\', text[match.end():])\n        if not number or Decimal(number.group(1)) != 0:\n            raise RuntimeError(\'实际应付金额非零或无法解析，禁止提交。\')\n    counts = []\n    for label in (\'本次扣除\', \'今日限量余额\', \'本日仍可导出\'):\n        found = re.findall(re.escape(label) + r\'\\s*[:：]?\\s*(\\d+)\\s*条\', text)\n        if len(found) != 1:\n            raise RuntimeError(f\'无法唯一读取{label}，禁止提交。\')\n        counts.append(int(found[0]))\n    count, balance, remaining = counts\n    if not 0 < count <= min(800, balance) or remaining != balance - count:\n        raise RuntimeError(\'条数超额或余额计算不一致，禁止提交。\')\n    return counts\n\n\ndef xlsx_rows(path):\n    """验证完整 XLSX 并读取首个工作表非空行数；不依赖 Excel 软件。"""\n    import xml.etree.ElementTree as ET\n    with ZipFile(path) as archive:\n        if not {\'[Content_Types].xml\', \'xl/workbook.xml\'}.issubset(archive.namelist()):\n            raise ValueError(\'不是 XLSX\')\n        if archive.testzip() is not None:\n            raise ValueError(\'ZIP 校验失败\')\n        sheets = sorted(n for n in archive.namelist() if re.fullmatch(r\'xl/worksheets/sheet\\d+\\.xml\', n))\n        if not sheets:\n            raise ValueError(\'没有工作表\')\n        root = ET.fromstring(archive.read(sheets[0]))\n        ns = {\'s\': \'http://schemas.openxmlformats.org/spreadsheetml/2006/main\'}\n        rows = root.findall(\'.//s:sheetData/s:row\', ns)\n        total = sum(1 for row in rows\n                   if any(c.find(\'s:v\', ns) is not None or c.find(\'s:is\', ns) is not None\n                          for c in row.findall(\'s:c\', ns)))\n        # 高级字段包已现场确认使用两行表头；统一返回“数据行+1”。\n        strings = []\n        if \'xl/sharedStrings.xml\' in archive.namelist():\n            strings = [\'\'.join(e.itertext()) for e in ET.fromstring(\n                archive.read(\'xl/sharedStrings.xml\')).findall(\'s:si\', ns)]\n        def values(row):\n            result = []\n            for cell in row.findall(\'s:c\', ns):\n                v = cell.find(\'s:v\', ns)\n                if v is not None and v.text is not None:\n                    result.append(strings[int(v.text)] if cell.get(\'t\') == \'s\' else v.text)\n                inline = cell.find(\'s:is\', ns)\n                if inline is not None:\n                    result.append(\'\'.join(inline.itertext()))\n            return result\n        if len(rows) >= 2:\n            first, second = values(rows[0]), values(rows[1])\n            if \'采购单位信息\' in first and \'单位名称\' in second and \'联系人\' in second:\n                total -= 1\n        return total\n\n\nclass OneDay:\n    def __init__(self, driver, day, base, confirm):\n        self.driver, self.day, self.confirm = driver, date.fromisoformat(day).isoformat(), confirm\n        self.base = Path(base)\n        self.base.mkdir(parents=True, exist_ok=True)\n        self.state_path = self.base / f\'one_day_{self.day}.json\'\n        self.lock_path = self.base / \'one_day.lock\'\n        self.state = {}\n\n    def save(self, **changes):\n        self.state.update(changes)\n        temp = self.state_path.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(self.state, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        temp.replace(self.state_path)\n\n    def body(self):\n        return self.driver.find_element(By.TAG_NAME, \'body\').text\n\n    def unique(self, by, locator):\n        matches = [e for e in self.driver.find_elements(by, locator) if e.is_displayed()]\n        if len(matches) != 1:\n            raise RuntimeError(f\'控件不是唯一匹配（{len(matches)}）：{locator}\')\n        return matches[0]\n\n    def text_click(self, text):\n        element = self.unique(By.XPATH, f"//*[normalize-space(text())=\'{text}\']")\n        self.visible_click(element)\n\n    def export_entry(self):\n        """区分固定结果栏与页面内副本，并合并同一按钮的内部文字。"""\n        # 已由现场诊断确认：底部栏含数量、修改条件和正式导出按钮。\n        footers = [e for e in self.driver.find_elements(By.CSS_SELECTOR, \'.data-footer-main\')\n                   if e.is_displayed()]\n        if len(footers) > 1:\n            # 现场确认顶部/底部两个完全相同的结果栏；先核对所有副本条数一致。\n            for footer in footers:\n                nums = footer.find_elements(By.CSS_SELECTOR, \'.dataExNum\')\n                if len(nums) != 1 or nums[0].text.strip() != str(self.state[\'count\']):\n                    raise RuntimeError(\'多个结果栏数量不一致，停止。\')\n            footers = [footers[-1]]\n        if len(footers) == 1:\n            footer = footers[0]\n            counts = [e for e in footer.find_elements(By.CSS_SELECTOR, \'.dataExNum\') if e.is_displayed()]\n            if len(counts) != 1 or counts[0].text.strip() != str(self.state[\'count\']):\n                raise RuntimeError(\'底部导出栏条数与本次查询不一致，尚未点击。\')\n            buttons = [e for e in footer.find_elements(By.CSS_SELECTOR, \'.data-now-export.dataBtnCom\')\n                       if e.is_displayed() and e.is_enabled() and e.text.strip() == \'立即导出\']\n            if len(buttons) != 1:\n                raise RuntimeError(\'底部正式导出按钮不唯一，尚未点击。\')\n            self.visible_click(buttons[0])\n            return\n        xpath = ".//*[normalize-space(text())=\'立即导出\']"\n\n        def controls(scope):\n            found = {}\n            for element in scope.find_elements(By.XPATH, xpath):\n                if not element.is_displayed():\n                    continue\n                # span等文字节点优先归到最近的真实交互元素。\n                owners = element.find_elements(By.XPATH,\n                    "ancestor-or-self::*[self::button or self::a or @role=\'button\'][1]")\n                owner = owners[0] if owners else element\n                if owner.is_displayed() and owner.is_enabled():\n                    found[owner.id] = owner\n            values = list(found.values())\n            # 无语义标签时，同一嵌套控件只保留最内层文字元素。\n            return [e for e in values if not any(\n                e.id != other.id and self.driver.execute_script(\n                    \'return arguments[0].contains(arguments[1]);\', e, other)\n                for other in values)]\n\n        for selector in (\'.dataExport_option.data_fixed\', \'.dataExport_option\'):\n            choices = {}\n            for scope in self.driver.find_elements(By.CSS_SELECTOR, selector):\n                if scope.is_displayed():\n                    for element in controls(scope):\n                        choices[element.id] = element\n            if len(choices) == 1:\n                self.visible_click(next(iter(choices.values())))\n                return\n\n        choices = controls(self.driver.find_element(By.TAG_NAME, \'body\'))\n        if len(choices) == 1:\n            self.visible_click(choices[0])\n            return\n        diagnostic = [e.find_element(By.XPATH, \'..\').get_attribute(\'outerHTML\')[:8000]\n                      for e in choices]\n        path = self.base / \'export_entry_diagnostic.json\'\n        path.write_text(json.dumps(diagnostic, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        raise RuntimeError(f\'导出入口仍有{len(choices)}个候选，尚未点击；控件结构已保存到{path}\')\n\n    def visible_click(self, element):\n        """先滚动并检查真实命中位置；不执行JS click，不重复提交点击。"""\n        self.driver.execute_script(\n            "arguments[0].scrollIntoView({block:\'center\',inline:\'center\',behavior:\'instant\'});", element)\n\n        def uncovered(_):\n            if not element.is_displayed() or not element.is_enabled():\n                return False\n            return self.driver.execute_script("""\n                const e = arguments[0], r = e.getBoundingClientRect();\n                const left = Math.max(0, r.left), right = Math.min(innerWidth, r.right);\n                const top = Math.max(0, r.top), bottom = Math.min(innerHeight, r.bottom);\n                if (right <= left || bottom <= top) return false;\n                const hit = document.elementFromPoint((left+right)/2, (top+bottom)/2);\n                return !!hit && (hit === e || e.contains(hit));\n            """, element)\n\n        # 少数固定栏较宽：通过正常滚动把控件放到不同高度。\n        for offset in (0, -180, 360):\n            if offset:\n                self.driver.execute_script(\'window.scrollBy(0, arguments[0]);\', offset)\n            try:\n                WebDriverWait(self.driver, 3, poll_frequency=0.2).until(uncovered)\n                break\n            except TimeoutException:\n                continue\n        else:\n            raise RuntimeError(\'控件仍被固定栏或弹窗遮挡，已停止，未强制点击。\')\n        element.click()\n\n    def filter_submit(self):\n        """只选与“重置”同组的确定，排除日期行上的确定。"""\n        resets = [e for e in self.driver.find_elements(By.XPATH, "//button[normalize-space(.)=\'重置\']")\n                  if e.is_displayed()]\n        if len(resets) != 1:\n            raise RuntimeError(\'无法唯一定位筛选区的重置按钮，未提交查询。\')\n        ancestor = resets[0]\n        for _ in range(6):\n            ancestor = ancestor.find_element(By.XPATH, \'..\')\n            confirms = [e for e in ancestor.find_elements(By.XPATH, ".//button[normalize-space(.)=\'确定\']")\n                        if e.is_displayed()]\n            if not confirms:\n                continue\n            if len(confirms) != 1:\n                raise RuntimeError(\'与重置同组的确定按钮仍不唯一，未提交查询。\')\n            if not confirms[0].is_enabled():\n                raise RuntimeError(\'筛选确定按钮不可用，未提交查询。\')\n            self.visible_click(confirms[0])\n            return\n        raise RuntimeError(\'未找到与重置同组的确定按钮，未提交查询。\')\n\n    def find_page(self, predicate, timeout=25, window_filter=None):\n        """重新枚举窗口及 iframe，避免沿用已关闭窗口；不导航、不提交。"""\n        def walk(depth=0):\n            if predicate():\n                return True\n            if depth >= 3:\n                return False\n            for frame in self.driver.find_elements(By.CSS_SELECTOR, \'iframe\'):\n                self.driver.switch_to.frame(frame)\n                if walk(depth + 1):\n                    return True\n                self.driver.switch_to.parent_frame()\n            return False\n\n        def probe(_):\n            handles = list(self.driver.window_handles)\n            try:\n                current = self.driver.current_window_handle\n                handles = [current] + [h for h in handles if h != current]\n            except NoSuchWindowException:\n                pass\n            for handle in handles:\n                try:\n                    if window_filter is not None and not window_filter(handle):\n                        continue\n                    self.driver.switch_to.window(handle)\n                    self.driver.switch_to.default_content()\n                    if walk():\n                        return True\n                except (NoSuchWindowException, StaleElementReferenceException):\n                    continue\n            return False\n        WebDriverWait(self.driver, timeout, poll_frequency=0.5).until(probe)\n\n    def filter_page(self):\n        predicate = lambda: \'筛选日期\' in self.body() and \'关键词匹配方式\' in self.body()\n        handle = getattr(self.driver, \'_jianyu_filter_handle\', None)\n        if handle not in self.driver.window_handles:\n            # 首次接入只查找已有筛选页，不创建标签页。\n            try:\n                self.find_page(predicate, timeout=15)\n                self.driver._jianyu_filter_handle = self.driver.current_window_handle\n                return\n            except TimeoutException:\n                handle = self.driver.current_window_handle\n        self.driver._jianyu_filter_handle = handle\n        self.driver.switch_to.window(handle)\n        self.driver.switch_to.default_content()\n        try:\n            self.find_page(predicate, timeout=15, window_filter=lambda h: h == handle)\n        except TimeoutException:\n            self.driver.switch_to.window(handle)\n            self.driver.get(FILTER_URL)\n            self.find_page(predicate, timeout=30, window_filter=lambda h: h == handle)\n\n    def release_order_tab(self):\n        """仅在预览结束或下载校验成功后关闭本次新建订单标签页。"""\n        order = getattr(self, \'_order_window\', None)\n        source = getattr(self, \'_order_source\', None)\n        try:\n            handles = self.driver.window_handles\n            if (getattr(self, \'_order_created\', False) and order in handles\n                    and source in handles and order != source):\n                self.driver.switch_to.window(order)\n                self.driver.close()\n            if source in self.driver.window_handles:\n                self.driver.switch_to.window(source)\n                self.driver.switch_to.default_content()\n        except Exception as error:\n            # 清理失败不改变已完成下载状态，也不触发重新扣除。\n            print(\'临时订单页未清理，请手动关闭：\', type(error).__name__)\n\n    def date_inputs(self):\n        candidates = []\n        for e in self.driver.find_elements(By.CSS_SELECTOR, \'input\'):\n            value = e.get_attribute(\'value\') or \'\'\n            if e.is_displayed() and re.fullmatch(r\'\\d{4}(?:年\\d{2}月\\d{2}日|[-/]\\d{2}[-/]\\d{2})\', value):\n                candidates.append(e)\n        if len(candidates) != 2:\n            raise RuntimeError(\'未找到唯一的一对日期输入框，需要核实日期控件。\')\n        return candidates\n\n    def calendar_panel(self):\n        panels = [e for e in self.driver.find_elements(By.CSS_SELECTOR, \'.el-picker-panel.el-date-picker\')\n                  if e.is_displayed()]\n        return panels[0] if len(panels) == 1 else False\n\n    def find_frame_here(self, predicate, timeout=8):\n        """仅在当前窗口遍历frame；My97弹层可能是筛选frame的兄弟。"""\n        def walk(depth=0):\n            if predicate():\n                return True\n            if depth >= 4:\n                return False\n            for frame in self.driver.find_elements(By.CSS_SELECTOR, \'iframe, frame\'):\n                if not frame.is_displayed():\n                    continue\n                self.driver.switch_to.frame(frame)\n                if walk(depth + 1):\n                    return True\n                self.driver.switch_to.parent_frame()\n            return False\n\n        def probe(_):\n            self.driver.switch_to.default_content()\n            try:\n                return walk()\n            except StaleElementReferenceException:\n                return False\n        WebDriverWait(self.driver, timeout, poll_frequency=0.2).until(probe)\n\n    def restore_date_frame(self):\n        def has_dates():\n            try:\n                return len(self.date_inputs()) == 2\n            except RuntimeError:\n                return False\n        self.find_frame_here(has_dates)\n\n    def my97_month(self):\n        months = set()\n        for cell in self.driver.find_elements(By.CSS_SELECTOR, \'.WdayTable td[onclick]\'):\n            classes = cell.get_attribute(\'class\') or \'\'\n            # 邻月补位日期可能也可点击，不能用它们判断当前月份。\n            if \'WotherDay\' in classes or \'WinvalidDay\' in classes:\n                continue\n            match = re.fullmatch(r\'\\s*day_Click\\(\\s*(\\d{4}),\\s*(\\d{1,2}),\\s*(\\d{1,2})\\s*\\);?\\s*\', cell.get_attribute(\'onclick\') or \'\')\n            if match:\n                months.add((int(match.group(1)), int(match.group(2))))\n        if len(months) != 1:\n            raise RuntimeError(\'无法从My97日历确定当前年月，停止。\')\n        return months.pop()\n\n    def my97_select_open_calendar(self):\n        target = date.fromisoformat(self.day)\n        for _ in range(240):\n            year, month = self.my97_month()\n            delta = (target.year - year) * 12 + target.month - month\n            if delta == 0:\n                break\n            selector = \'#dpTitle .NavImgl a\' if delta < 0 else \'#dpTitle .NavImgr a\'\n            self.unique(By.CSS_SELECTOR, selector).click()\n            WebDriverWait(self.driver, 5).until(lambda _: self.my97_month() != (year, month))\n        else:\n            raise RuntimeError(\'My97日历翻页超过240个月，停止。\')\n\n        cells = []\n        for cell in self.driver.find_elements(By.CSS_SELECTOR, \'.WdayTable td[onclick]\'):\n            match = re.fullmatch(r\'\\s*day_Click\\(\\s*(\\d{4}),\\s*(\\d{1,2}),\\s*(\\d{1,2})\\s*\\);?\\s*\', cell.get_attribute(\'onclick\') or \'\')\n            if (match and tuple(map(int, match.groups())) == (target.year, target.month, target.day)\n                    and cell.is_displayed() and \'WinvalidDay\' not in (cell.get_attribute(\'class\') or \'\')):\n                cells.append(cell)\n        if len(cells) != 1:\n            raise RuntimeError(f\'My97目标日期{self.day}不可选或不唯一，停止。\')\n        # 用真实鼠标点击，不调用网页内部day_Click函数。\n        cells[0].click()\n        for button in self.driver.find_elements(By.CSS_SELECTOR, \'#dpOkInput\'):\n            if button.is_displayed() and button.is_enabled():\n                button.click()\n                break\n\n    def calendar_select(self, element):\n        """优先识别现场确认的My97日历；不修改readonly或直接注入日期值。"""\n        target = date.fromisoformat(self.day)\n        # 上一笔查询/下载完成后页面可能仍显示加载遮罩。先等它消失，避免点击被拦截。\n        WebDriverWait(self.driver, 60, poll_frequency=0.5).until(\n            lambda _: not any(e.is_displayed() for e in self.driver.find_elements(\n                By.CSS_SELECTOR, \'.loading_\')))\n        element.click()\n        try:\n            self.find_frame_here(lambda: any(e.is_displayed() for e in self.driver.find_elements(By.CSS_SELECTOR, \'.WdateDiv\')), timeout=10)\n        except TimeoutException:\n            self.restore_date_frame()\n        else:\n            try:\n                self.my97_select_open_calendar()\n            finally:\n                self.restore_date_frame()\n            return\n        try:\n            panel = WebDriverWait(self.driver, 8).until(lambda _: self.calendar_panel())\n        except TimeoutException:\n            # 保存真实DOM用于适配其他日历，不能猜测点击后继续扣额度。\n            (self.base / \'calendar_diagnostic.html\').write_text(self.driver.page_source, encoding=\'utf-8\')\n            raise RuntimeError(\'未识别到 Element UI 单日历；已保存 calendar_diagnostic.html，尚未提交。\')\n\n        def shown_month():\n            current = self.calendar_panel()\n            if not current:\n                raise RuntimeError(\'日历意外关闭，停止。\')\n            header = current.find_element(By.CSS_SELECTOR, \'.el-date-picker__header\')\n            return calendar_month(header.text)\n\n        for _ in range(240):\n            year, month = shown_month()\n            delta = (target.year - year) * 12 + target.month - month\n            if delta == 0:\n                break\n            panel = self.calendar_panel()\n            selector = \'.el-date-picker__prev-btn.el-icon-arrow-left\' if delta < 0 else \'.el-date-picker__next-btn.el-icon-arrow-right\'\n            controls = [e for e in panel.find_elements(By.CSS_SELECTOR, selector) if e.is_displayed() and e.is_enabled()]\n            if len(controls) != 1:\n                raise RuntimeError(\'未唯一找到日历上月/下月按钮，尚未提交。\')\n            controls[0].click()\n            WebDriverWait(self.driver, 5).until(lambda _: shown_month() != (year, month))\n        else:\n            raise RuntimeError(\'日历翻页超过240个月，停止。\')\n\n        # 排除前后月份补位日期及不可选日期。\n        panel = self.calendar_panel()\n        cells = [e for e in panel.find_elements(By.CSS_SELECTOR,\n                 \'.el-date-table td:not(.prev-month):not(.next-month):not(.disabled):not(.week)\')\n                 if e.is_displayed() and e.text.strip() == str(target.day)]\n        if len(cells) != 1:\n            raise RuntimeError(f\'目标日期 {self.day} 不可选或不唯一，尚未提交。\')\n        cells[0].click()\n\n    def dates(self):\n        # 页面异步重绘会使旧 input 引用失效；仅重试日期设置，不重试查询/扣额。\n        for attempt in range(3):\n            try:\n                if attempt:\n                    # 日历组件偶尔在连续查询后不弹出。刷新本筛选页再重新定位；此时尚未创建订单。\n                    self.driver.switch_to.default_content()\n                    self.driver.get(FILTER_URL)\n                    self.find_page(lambda: \'筛选日期\' in self.body() and \'关键词匹配方式\' in self.body(),\n                                   timeout=30,\n                                   window_filter=lambda h: h == self.driver._jianyu_filter_handle)\n                return self._set_dates_once()\n            except (StaleElementReferenceException, TimeoutException, RuntimeError) as error:\n                retryable = (isinstance(error, (StaleElementReferenceException, TimeoutException))\n                             or str(error).startswith(\'未识别到 Element UI 单日历\'))\n                if not retryable:\n                    raise\n                if attempt == 2:\n                    raise RuntimeError(\'日期控件连续三次未能打开，已停止且未提交。\') from error\n                print(f\'日期控件未就绪，刷新筛选页后重试（{attempt + 2}/3）\', flush=True)\n                time.sleep(1)\n\n    def _set_dates_once(self):\n        candidates = self.date_inputs()\n        # 往后移动时先改结束日期，往前移动时先改开始日期，避免范围约束。\n        current_end = date(*map(int, re.findall(r\'\\d+\', candidates[1].get_attribute(\'value\'))))\n        order = (1, 0) if date.fromisoformat(self.day) > current_end else (0, 1)\n        for index in order:\n            element = self.date_inputs()[index]\n            old = element.get_attribute(\'value\')\n            target = (date.fromisoformat(self.day).strftime(\'%Y年%m月%d日\') if \'年\' in old\n                      else self.day.replace(\'-\', \'/\') if \'/\' in old else self.day)\n            if old == target:\n                continue\n            if element.get_attribute(\'readonly\'):\n                self.calendar_select(element)\n            else:\n                element.click()\n                element.send_keys(Keys.CONTROL, \'a\')\n                element.send_keys(target)\n                element.send_keys(Keys.TAB)\n            WebDriverWait(self.driver, 8).until(\n                lambda _, i=index, t=target: self.date_inputs()[i].get_attribute(\'value\') == t)\n            # 自动关闭仍打开的弹层，不点页面其他按钮。\n            # 日期弹层由选日/确定关闭，避免给只读输入框发送按键触发其他事件。\n        candidates = self.date_inputs()\n        if any(date(*map(int, re.findall(r\'\\d+\', e.get_attribute(\'value\')))).isoformat() != self.day for e in candidates):\n            raise RuntimeError(\'开始和结束日期未同时设为目标日期，停止。\')\n        print(f\'起止日期已核实：{self.day}\', flush=True)\n        return candidates\n\n    def no_data_visible(self):\n        text = self.body()\n        return (\'未匹配到数据\' in text\n                and \'对不起，没有匹配到数据，请修改数据导出条件\' in text)\n\n    def close_no_data(self):\n        if self.no_data_visible():\n            self.text_click(\'立即修改\')\n            WebDriverWait(self.driver, 15).until(lambda _: not self.no_data_visible())\n\n    def query_filters(self, select_regions=None, wait_query=None):\n        self.close_no_data()\n        self.filter_page()\n        inputs = self.dates()\n        if select_regions:\n            select_regions(self)\n        else:\n            self.visible_click(self.unique(By.XPATH, "//span[contains(@class,\'select-area\') and normalize-space(.)=\'全国\']"))\n        text = self.body()\n        # 根据已经读出的关键词标签格式核对，不用全文包含判断，避免说明文字误命中。\n        actual = set(re.findall(r\'关键词\\s*[:：]\\s*([^\\s]+)\', text.split(\'例：\')[0]))\n        if actual != WORDS:\n            raise RuntimeError(f\'关键词不是指定的四项：{actual}。请在页面配置后再运行。\')\n        checks = self.driver.find_elements(By.CSS_SELECTOR, \'input.el-checkbox__original[type="checkbox"]\')\n        matches = [e for e in checks if e.get_attribute(\'value\') in {\'1\', \'2\', \'3\', \'4\'}]\n        if len(matches) != 4 or {e.get_attribute(\'value\') for e in matches} != {\'1\', \'2\', \'3\', \'4\'}:\n            raise RuntimeError(\'未唯一找到四项关键词匹配方式。\')\n        for checkbox in matches:\n            if not checkbox.is_selected():\n                self.visible_click(checkbox.find_element(By.XPATH, "ancestor::label[1]").find_element(By.CSS_SELECTOR, \'.el-checkbox__inner\'))\n        if not all(e.is_selected() for e in matches):\n            raise RuntimeError(\'匹配方式未全部勾选。\')\n        print(f\'已设置 {self.day} 及四项匹配方式；四个关键词核对通过。\', flush=True)\n        self.filter_submit()\n        if wait_query:\n            wait_query(self)\n        # 仅进入订单预览；最终提交前再核对订单条数。\n        WebDriverWait(self.driver, 25).until(lambda _: self.no_data_visible() or re.search(r\'为您筛选到\\s*\\d+\\s*条数据\', self.body()))\n        if self.no_data_visible():\n            self.close_no_data()\n            self.save(count=0, status=\'filtered\')\n            return 0\n        found = set(map(int, re.findall(r\'为您筛选到\\s*(\\d+)\\s*条数据\', self.body())))\n        if len(found) != 1:\n            raise RuntimeError(\'查询条数不一致，停止。\')\n        count = found.pop()\n        self.save(count=count, status=\'filtered\')\n        return count\n\n    def configure(self):\n        count = self.query_filters()\n        if not 0 < count <= 800:\n            raise RuntimeError(f\'本日查询为{count}条；本测试仅支持1至800条，不拆省份。\')\n        self.open_order()\n\n    def open_order(self, prepare_payment=True):\n        count = self.state[\'count\']\n        if not 0 < count <= 800:\n            raise RuntimeError(\'只能为1至800条创建结算预览\')\n        print(f\'查询显示 {count} 条，进入订单页核对。\', flush=True)\n        source = self.driver.current_window_handle\n        before = set(self.driver.window_handles)\n        self._order_source = source\n        # 枚举窗口会丢失iframe上下文，恢复本次查询所在frame，不跳到旧页面。\n        self.find_frame_here(lambda: bool(re.search(\n            rf\'为您筛选到\\s*{count}\\s*条数据\', self.body())))\n        self.export_entry()\n\n        def new_order_window(handle):\n            return (handle == source or handle not in before)\n\n        def expected_order():\n            text = self.body()\n            url = self.driver.execute_script(\'return location.href\')\n            counts = re.findall(r\'已选择\\s*(\\d+)\\s*条数据\', text)\n            return (\'/front/dataExport/toCreateOrderPage/\' in url\n                    and \'选择支付方式\' in text and counts == [str(count)])\n\n        try:\n            self.find_page(expected_order, timeout=30, window_filter=new_order_window)\n        except TimeoutException:\n            raise RuntimeError(f\'未找到本次新开的{count}条订单页，未选择旧订单、未确认扣除。\')\n        self._order_window = self.driver.current_window_handle\n        self._order_created = self._order_window not in before\n        self._order_frame_url = self.driver.execute_script(\'return location.href\')\n        print(f\'已锁定本次订单页：{count}条\', flush=True)\n        # 余额探测只读取订单页数字。即使全国条数高于余额并弹出“余额不足”，\n        # 也不寻找协议、不确认扣除；调用方读取余额后关闭本次临时订单页。\n        if not prepare_payment:\n            return\n        self.text_click(\'单日限量数据包\')\n        WebDriverWait(self.driver, 15).until(lambda _: \'本次扣除\' in self.body())\n        # 附件中的真实协议 DOM：仅点击方框，禁止点协议链接。\n        labels = [e for e in self.driver.find_elements(By.XPATH, "//label[.//input[@type=\'checkbox\']]")\n                  if e.is_displayed() and \'已阅读并同意\' in e.text and \'服务条款\' in e.text]\n        if len(labels) != 1:\n            raise RuntimeError(\'协议复选框不唯一。\')\n        checkbox = labels[0].find_element(By.CSS_SELECTOR, \'input[type="checkbox"]\')\n        if not checkbox.is_selected():\n            self.visible_click(labels[0].find_element(By.CSS_SELECTOR, \'.el-checkbox__inner\'))\n        WebDriverWait(self.driver, 5).until(lambda _: checkbox.is_selected())\n\n    def verify(self):\n        if (hasattr(self, \'_order_window\') and\n                (self.driver.current_window_handle != self._order_window or\n                 self.driver.execute_script(\'return location.href\') != self._order_frame_url)):\n            raise RuntimeError(\'当前窗口已不是本次锁定订单，禁止扣除。\')\n        text = self.body()\n        chosen = re.findall(r\'已选择\\s*(\\d+)\\s*条数据\', text)\n        if chosen != [str(self.state[\'count\'])]:\n            raise RuntimeError(\'订单页顶部所选条数与本批次不同，禁止扣除。\')\n        cards = [e for e in self.driver.find_elements(By.CSS_SELECTOR, \'.spec-card.active\') if e.is_displayed()]\n        if len(cards) != 1 or \'单日限量数据包\' not in cards[0].text:\n            raise RuntimeError(\'支付方式不是单日限量数据包。\')\n        price = cards[0].find_element(By.CSS_SELECTOR, \'.spec-c-price-text\').get_attribute(\'textContent\').strip()\n        if Decimal(price) != 0:\n            raise RuntimeError(\'选中数据包不是0元。\')\n        if any(e.is_displayed() for e in self.driver.find_elements(By.XPATH, "//*[normalize-space(text())=\'确定支付\']")):\n            raise RuntimeError(\'仍有可见现金支付按钮，停止。\')\n        count, balance, remaining = order_counts(text)\n        if count != self.state[\'count\']:\n            raise RuntimeError(\'订单条数与查询条数不同，停止。\')\n        labels = [e for e in self.driver.find_elements(By.XPATH, "//label[.//input[@type=\'checkbox\']]")\n                  if e.is_displayed() and \'已阅读并同意\' in e.text and \'服务条款\' in e.text]\n        if len(labels) != 1 or not labels[0].find_element(By.CSS_SELECTOR, \'input[type="checkbox"]\').is_selected():\n            raise RuntimeError(\'协议未勾选，停止。\')\n        return count, balance, remaining\n\n    def download_complete(self, timeout=45):\n        folder = Path(self.state[\'download_dir\'])\n        deadline = time.monotonic() + timeout\n        while time.monotonic() < deadline:\n            for path in folder.glob(\'*.xlsx\'):\n                try:\n                    rows = xlsx_rows(path)\n                    if rows != self.state[\'count\'] + 1:\n                        raise RuntimeError(f\'文件有{rows}个非空行，与{self.state["count"]}条数据加表头不一致，保留待人工核对：{path}\')\n                    self.save(status=\'downloaded\', file=str(path))\n                    print(f\'完成，文件：{path}\', flush=True)\n                    return path\n                except (OSError, BadZipFile, ValueError):\n                    continue\n            time.sleep(1)\n        return None\n\n    def execute(self):\n        progress = self.base / \'range_progress.json\'\n        if progress.exists():\n            # 共用运行锁只阻止并发；还必须阻止不同入口顺序重复提交。\n            state = json.loads(progress.read_text(encoding=\'utf-8\'))\n            records = list(state.get(\'completed\', []))\n            if state.get(\'pending\'):\n                records.append(state[\'pending\'])\n            if any(record.get(\'date\') == self.day for record in records):\n                raise RuntimeError(\'该日期已有多日任务批次，请使用原多日入口恢复或继续，禁止整日重复导出。\')\n        if self.state_path.exists():\n            self.state = json.loads(self.state_path.read_text(encoding=\'utf-8\'))\n            if self.state.get(\'status\') == \'downloaded\':\n                print(\'该日期已完成，不重复扣除：\', self.state.get(\'file\'))\n                return self.state.get(\'file\')\n            if self.state.get(\'status\') in {\'submitting\', \'submitted\'}:\n                raise RuntimeError(\'该日期已尝试提交，请核对原导出记录并下载，不自动重试扣除。\')\n        folder = self.base / \'downloads\' / (self.day + \'_\' + uuid4().hex[:10])\n        folder.mkdir(parents=True, exist_ok=True)\n        self.save(date=self.day, status=\'started\', download_dir=str(folder))\n        self.driver.execute_cdp_cmd(\'Browser.setDownloadBehavior\', {\'behavior\': \'allow\', \'downloadPath\': str(folder.resolve())})\n        self.configure()\n        count, balance, remaining = self.verify()\n        print(f\'校验通过：免费扣除{count}条，余额{balance}，扣除后{remaining}。\', flush=True)\n        if not self.confirm:\n            print(\'预演结束，未确认扣除。\')\n            return None\n        return self.submit_order()\n\n    def submit_order(self):\n        self.verify()\n        button = self.unique(By.XPATH, "//button[normalize-space(.)=\'确认扣除\']")\n        if not button.is_enabled():\n            raise RuntimeError(\'确认扣除按钮不可用。\')\n        self.save(status=\'submitting\', submitted_at=datetime.now().isoformat(), order_url=self.driver.current_url)\n        button.click()  # 唯一会消耗额度的操作；异常后也不自动重试。\n        self.find_page(lambda: \'数据导出成功\' in self.body() and \'单日限量数据包扣除\' in self.body(), timeout=120)\n        self.save(status=\'submitted\')\n        result = self.download_complete(timeout=15)\n        if result:\n            return result\n        # 已知成功页链接，记录页的DOM尚未实际核实，不按日期猜测相同历史订单。\n        self.text_click(\'查看数据导出记录\')\n        print(\'已生成订单但未检测到文件。请在原记录中点击“点击下载”；程序继续等待120秒，不会再扣额度。\', flush=True)\n        result = self.download_complete(timeout=120)\n        if not result:\n            raise RuntimeError(\'下载未完成。订单状态已保留，请从原导出记录下载，不要重新提交。\')\n        return result\n\n\ndef export_one_day(driver, day=\'2025-01-02\', base=r\'D:\\桌面\\data\', confirm=False):\n    """confirm=False：运行至结算校验后停止；True：允许扣除免费条数。"""\n    job = OneDay(driver, day, base, confirm)\n    try:\n        with job.lock_path.open(\'x\', encoding=\'utf-8\') as lock:\n            lock.write(\'单日导出运行中；确认旧任务停止后方可删除此锁。\')\n    except FileExistsError:\n        raise RuntimeError(\'已有单日任务运行锁，不启动第二个任务。\')\n    try:\n        return job.execute()\n    except Exception as error:\n        print(f\'已停止：{error}\', flush=True)\n        try:\n            driver.save_screenshot(str(job.base / f\'one_day_error_{job.day}.png\'))\n        except Exception:\n            pass\n        raise\n    finally:\n        job.lock_path.unlink(missing_ok=True)\n', 'automatic_ui': '"""现场DOM验证的自动筛选适配器（2026-09-11）。"""\nimport re\nfrom selenium.webdriver.common.by import By\nfrom selenium.webdriver.support.ui import WebDriverWait\n\n\nclass AutomaticUI:\n    def balance(self, driver):\n        # 优先读取当前已打开的免费额度结算页；未打开时由后端只预览小批次。\n        for handle in driver.window_handles:\n            driver.switch_to.window(handle)\n            driver.switch_to.default_content()\n            text = driver.find_element(By.TAG_NAME, \'body\').text\n            if \'本次扣除\' in text and \'今日限量余额\' in text:\n                values = re.findall(r\'今日限量余额\\s*[:：]?\\s*(\\d+)\\s*条\', text)\n                if len(values) == 1:\n                    return int(values[0])\n        raise RuntimeError(\'尚无余额预览页\')\n\n    def regions(self, driver, day):\n        names = [e.text.strip() for e in driver.find_elements(\n            By.CSS_SELECTOR, \'#area-del + .select-area-box > span.select-area\') if e.is_displayed()]\n        names = [n for n in names if n != \'全国\']\n        if not names or len(set(names)) != len(names):\n            raise RuntimeError(\'当天地区列表为空或重复，停止避免漏数。\')\n        return names\n\n    def select_regions(self, job, regions):\n        d = job.driver\n        def selected():\n            return {e.text.strip() for e in d.find_elements(By.CSS_SELECTOR, \'#area-del .delete-close\')}\n        current = selected()\n        if not regions or \'全国\' in current:\n            job.visible_click(job.unique(By.XPATH, "//div[@id=\'area-del\']/following-sibling::div[contains(@class,\'select-area-box\')][1]/span[normalize-space(.)=\'全国\']"))\n            current = set()\n        # 已经选择全部/多数省份时，只删除差集，不重新添加所有省份。\n        for name in sorted(current - set(regions)):\n            chips = [e for e in d.find_elements(By.CSS_SELECTOR, \'#area-del .delete-close\') if e.text.strip() == name]\n            if len(chips) != 1:\n                raise RuntimeError(\'无法唯一定位待移除省份标签\')\n            job.visible_click(chips[0].find_element(By.CSS_SELECTOR, \'.icon-guanbi\'))\n        for name in [r for r in regions if r not in current]:\n            job.visible_click(job.unique(By.XPATH, f"//div[@id=\'area-del\']/following-sibling::div[contains(@class,\'select-area-box\')][1]/span[normalize-space(.)=\'{name}\']"))\n            popups = [e for e in d.find_elements(By.CSS_SELECTOR, \'.dialog\') if e.is_displayed()]\n            if popups:\n                popup = popups[0]\n                all_region = [e for e in popup.find_elements(By.CSS_SELECTOR, \'span.select-area\')\n                              if e.is_displayed() and e.text.strip() in {\'全省\', \'全市\', \'全部\'}]\n                if len(all_region) != 1:\n                    raise RuntimeError(f\'{name}的全省选项未唯一定位\')\n                job.visible_click(all_region[0])\n                job.visible_click(job.unique(By.CSS_SELECTOR, "button[onclick=\'areaSelect(true)\']"))\n        WebDriverWait(d, 8).until(lambda _: selected() == (set(regions) if regions else {\'全国\'}) or\n                                 (not regions and selected() == set()))\n        self.arm_query(job)\n\n    def arm_query(self, job):\n        # 同时观察请求和结果区域变化。部分页面请求不会经过 XMLHttpRequest，\n        # 此时用结果区 DOM 更新作为后备信号，避免把已完成查询误判为超时。\n        job.driver.execute_script(\'\'\'\n        if (!window.__autoQueryInstalled) {\n          window.__autoQueryInstalled=true;\n          const original=XMLHttpRequest.prototype.open;\n          XMLHttpRequest.prototype.open=function(method,url,...rest) {\n            const target=new URL(String(url),location.href).pathname===\'/front/dataExport/sieveData\';\n            const q=window.__autoQuery;\n            if(target && q) {\n              q.pending++;\n              this.addEventListener(\'loadend\',()=>{\n                if(window.__autoQuery===q) {q.pending--;q.done++;q.status=this.status;q.finished=Date.now();}\n              });\n            }\n            return original.call(this,method,url,...rest);\n          };\n        }\n        window.__autoQuery={done:0,status:0,finished:0,pending:0};\n        if (window.__autoResultObserver) window.__autoResultObserver.disconnect();\n        window.__autoResultMutation={count:0,last:0};\n        const resultRoot=document.querySelector(\'#dataExport_main\') || document.body;\n        window.__autoResultObserver=new MutationObserver((items)=>{\n          if(items.length) {\n            window.__autoResultMutation.count+=items.length;\n            window.__autoResultMutation.last=Date.now();\n          }\n        });\n        window.__autoResultObserver.observe(resultRoot,{subtree:true,childList:true,characterData:true,attributes:true});\n        \'\'\')\n\n    def wait_query(self, job):\n        def fresh(_):\n            q=job.driver.execute_script(\'return window.__autoQuery\')\n            if q and q[\'done\'] and q[\'status\'] != 200:\n                raise RuntimeError(\'本次筛选请求失败，未读取旧结果。\')\n            if q and q[\'done\'] and q[\'pending\']==0 and job.driver.execute_script(\'return Date.now()-window.__autoQuery.finished\') > 500:\n                return True\n            # 请求监听漏报时，必须确认结果区确实发生过变化、页面已稳定且\n            # 已出现新的结果数量或“无数据”提示，不能直接读取旧表格。\n            return bool(job.driver.execute_script(\'\'\'\n              const m=window.__autoResultMutation;\n              if(!m || !m.count || Date.now()-m.last<800) return false;\n              const loading=[...document.querySelectorAll(\'.el-loading-mask,.loading_,.loading\')]\n                .some(e=>{const s=getComputedStyle(e);return s.display!==\'none\'&&s.visibility!==\'hidden\'&&e.offsetParent!==null;});\n              if(loading) return false;\n              const text=document.body.innerText||\'\';\n              return /为您筛选到\\\\s*\\\\d+\\\\s*条数据/.test(text) || /暂无数据|没有数据/.test(text);\n            \'\'\'))\n        WebDriverWait(job.driver, 90, poll_frequency=0.2).until(fresh)\n\n\nclass DeferredNotifier:\n    """未授权真实邮件时只保留待发送记录。"""\n    def send(self, record):\n        raise RuntimeError(\'尚未启用真实邮件发送\')\n', 'selenium_range': '"""复用原单日导出。未知页面控件通过交互确认或显式 UI 适配器操作。"""\nimport json\nimport re\nfrom pathlib import Path\n\nfrom export_one_day import OneDay, FILTER_URL, xlsx_rows\nfrom export_range import ExportRange\nfrom email_notice import SMTPNotifier\nfrom automatic_ui import AutomaticUI, DeferredNotifier\n\n\nclass InteractiveUI:\n    """可立即使用的有人值守模式，不猜测网站尚未采集的省份 DOM。\n    自动模式可传入具有同名方法的 UI 对象，方法必须核对实际选中状态。\n    """\n    def balance(self, driver):\n        return int(input(\'请在网站核对【今日限量余额】，输入实际剩余条数（0～800）：\').strip())\n\n    def regions(self, driver, day):\n        names = input(\'请从网站提供完整的省级地区清单（含港澳台等实际选项），用中文逗号或英文逗号分隔：\')\n        return [s.strip() for s in names.replace(\'，\', \',\').split(\',\') if s.strip()]\n\n    def select_regions(self, job, regions):\n        target = \'全国\' if not regions else \'、\'.join(regions)\n        print(f\'请在当前筛选页仅选中【{target}】，清除其他地区；其他筛选保持全部、单位输入框留空。\')\n        if input(f\'核对完成后原样输入“{target}”：\').strip() != target:\n            raise RuntimeError(\'地区未确认，未执行查询\')\n\n    def wait_query(self, job):\n        if input(\'请等待本次筛选结果加载完成，确认日期、地区正确后输入“完成”：\').strip() != \'完成\':\n            raise RuntimeError(\'新查询结果未确认\')\n\n\nclass SeleniumBackend:\n    def __init__(self, driver, base, ui=None):\n        self.driver, self.base = driver, Path(base)\n        self.ui = ui or AutomaticUI()\n        self.job, self.selection = None, None\n        self.last_order_check = None\n        self.balance_day = \'2025-01-03\'\n\n    def balance(self):\n        if not isinstance(self.ui, AutomaticUI):\n            return self.ui.balance(self.driver)\n        # 新建订单预览读取实时余额，不使用昨天停留页面中的旧余额。\n        day = self.balance_day\n        count = self.query(day, [])\n        if not 0 < count <= 800:\n            for region in self.regions(day):\n                count = self.query(day, [region])\n                if 0 < count <= 800:\n                    break\n            else:\n                raise RuntimeError(\'无法生成可读余额的免费额度预览，未提交。\')\n        self.job.open_order(prepare_payment=False)\n        balances = re.findall(r\'今日限量余额\\s*[:：]?\\s*(\\d+)\\s*条\', self.job.body())\n        if len(balances) != 1:\n            raise RuntimeError(\'本次余额预览页条数不唯一\')\n        balance = int(balances[0])\n        self.job.release_order_tab()\n        return balance\n\n    def existing_day(self, day):\n        path = self.base / f\'one_day_{day}.json\'\n        if not path.exists():\n            return None\n        state = json.loads(path.read_text(encoding=\'utf-8\'))\n        if state.get(\'status\') == \'submitted\' and state.get(\'download_dir\'):\n            candidates = list(Path(state[\'download_dir\']).glob(\'*.xlsx\'))\n            valid = [p for p in candidates if xlsx_rows(p) == state[\'count\'] + 1]\n            if len(valid) == 1:\n                state.update(status=\'downloaded\', file=str(valid[0]))\n                temp = path.with_suffix(\'.tmp\')\n                temp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n                temp.replace(path)\n        if state.get(\'status\') in {\'submitting\', \'submitted\'}:\n            raise RuntimeError(f\'{day} 存在旧版未完成订单，请先恢复原下载，不再扣额\')\n        if state.get(\'status\') != \'downloaded\':\n            return None\n        if xlsx_rows(state[\'file\']) != state[\'count\'] + 1:\n            raise RuntimeError(f\'{day} 旧导出文件校验失败\')\n        return state\n\n    def regions(self, day):\n        return self.ui.regions(self.driver, day)\n\n    def query(self, day, regions):\n        # 同一天复用筛选页，省份查询不再反复导航或进入订单页。\n        if self.job is not None and self.job.day == day:\n            job = self.job\n        else:\n            job = OneDay(self.driver, day, self.base / \'batch_state\', False)\n            job.state_path = self.base / \'batch_state\' / \'preview.json\'\n        self.job, self.selection = job, (day, list(regions))\n        return job.query_filters(lambda j: self.ui.select_regions(j, regions), self.ui.wait_query)\n\n    def prepare(self, day, regions, count):\n        self.last_order_check = None\n        if self.selection != (day, list(regions)) or self.job.state.get(\'count\') != count:\n            raise RuntimeError(\'结算与最后一次查询条件不符\')\n        self.job.open_order()\n        balances = re.findall(r\'今日限量余额\\s*[:：]?\\s*(\\d+)\\s*条\', self.job.body())\n        if len(balances) != 1:\n            raise RuntimeError(\'订单的实际余额不唯一\')\n        balance = int(balances[0])\n        if balance < count:\n            # 不足额度的预览不提交，交给调度器缩小地区范围。\n            self.last_order_check = {\n                \'date\': day, \'regions\': list(regions), \'count\': count,\n                \'today_balance\': balance, \'after_export\': None,\n                \'allowed\': False, \'reason\': \'insufficient_balance\'}\n            print(f\'提交前额度核对：本次扣除{count}条；今日限量余额{balance}条；\'\n                  \'本日仍可导出无法生成；不允许导出，重新拆分。\', flush=True)\n            self.job.release_order_tab()\n            return balance, count\n        actual_count, balance, after = self.job.verify()\n        allowed = (actual_count == count and actual_count <= balance\n                   and after == balance - actual_count)\n        self.last_order_check = {\n            \'date\': day, \'regions\': list(regions), \'count\': actual_count,\n            \'today_balance\': balance, \'after_export\': after,\n            \'allowed\': allowed, \'reason\': \'verified\' if allowed else \'inconsistent\'}\n        decision = \'允许导出\' if allowed else \'禁止导出\'\n        print(f\'提交前额度核对：本次扣除{actual_count}条；今日限量余额{balance}条；\'\n              f\'本日仍可导出{after}条；{decision}\', flush=True)\n        if not allowed:\n            raise RuntimeError(\'最终订单额度核对不一致，禁止提交\')\n        return balance, actual_count\n\n    def submit(self, batch_id):\n        job = self.job\n        folder = self.base / \'downloads\' / f\'{job.day}_{batch_id}\'\n        folder.mkdir(parents=True, exist_ok=True)\n        job.state_path = self.base / \'batch_state\' / f\'{batch_id}.json\'\n        if job.state_path.exists():\n            raise RuntimeError(\'批次已存在，禁止重复提交\')\n        job.save(date=job.day, regions=self.selection[1], batch_id=batch_id,\n                 download_dir=str(folder.resolve()), status=\'prepared\')\n        self.driver.execute_cdp_cmd(\'Browser.setDownloadBehavior\', {\n            \'behavior\': \'allow\', \'downloadPath\': str(folder.resolve())})\n        file = job.submit_order()\n        job.release_order_tab()\n        return file\n\n    def recover(self, pending):\n        path = self.base / \'batch_state\' / f"{pending[\'batch_id\']}.json"\n        if not path.exists():\n            raise RuntimeError(\'提交状态不确定，请核对网站原记录；不会创建第二笔订单\')\n        state = json.loads(path.read_text(encoding=\'utf-8\'))\n        if (state.get(\'count\') != pending[\'count\'] or state.get(\'date\') != pending[\'date\']\n                or state.get(\'regions\') != pending[\'regions\']):\n            raise RuntimeError(\'批次记录与调度进度不一致\')\n        job = OneDay(self.driver, pending[\'date\'], self.base / \'batch_state\', False)\n        job.state_path, job.state = path, state\n        if state.get(\'status\') == \'downloaded\':\n            file = Path(state[\'file\'])\n            if xlsx_rows(file) != pending[\'count\'] + 1:\n                raise RuntimeError(\'原文件校验失败\')\n            return file\n        folder = Path(state[\'download_dir\']).resolve()\n        self.driver.execute_cdp_cmd(\'Browser.setDownloadBehavior\', {\n            \'behavior\': \'allow\', \'downloadPath\': str(folder)})\n        print(f\'仅恢复原批次 {pending["batch_id"]}，日期 {pending["date"]}，地区 {pending["regions"]}。\')\n        print(f\'请核对网站原订单并下载至 {folder}；未查到原订单请停止，不要重新提交。\')\n        return job.download_complete(timeout=120)\n\n\ndef export_range(driver, base=r\'D:\\桌面\\data\', start=\'2025-01-01\', end=None,\n                 confirm=False, ui=None, notifier=None, daily_limit=200):\n    """end=None：本次运行当天；每日重新调用即从原进度继续。"""\n    if confirm and notifier is None:\n        notifier = DeferredNotifier()\n    backend = SeleniumBackend(driver, base, ui)\n    backend.balance_day = start\n    return ExportRange(backend, base, start, end,\n                       confirm, notifier=notifier, daily_limit=daily_limit).run()\n\n\ndef run_daily(driver, **kwargs):\n    """持续运行到完成；额度不足时等待中国时区次日00:01，再读取实际余额。\n    浏览器失效等异常直接抛出；重新连接 driver 后再次调用即可恢复。\n    """\n    import time\n    from datetime import datetime, timedelta, timezone\n    while True:\n        state = export_range(driver, **kwargs)\n        if state[\'status\'] != \'waiting_quota\':\n            return state\n        now = datetime.now(timezone(timedelta(hours=8)))\n        tomorrow = (now + timedelta(days=1)).replace(hour=0, minute=1, second=0, microsecond=0)\n        print(f\'当日额度不足，保留当前日期与省份；将在 {tomorrow.isoformat()} 恢复。\')\n        while datetime.now(tomorrow.tzinfo) < tomorrow:\n            time.sleep(min(30, max(0.1, (tomorrow-datetime.now(tomorrow.tzinfo)).total_seconds())))\n', 'fast_range': '"""同页查询、100条测试上限；不在导入时启动浏览器或提交。"""\nfrom datetime import date, timedelta\nfrom pathlib import Path\nfrom uuid import uuid4\nfrom export_range import ExportRange, province_key\n\n\nclass FastRange(ExportRange):\n    def __init__(self, *args, strategy=\'prefix\', **kwargs):\n        if strategy != \'prefix\':\n            raise ValueError(\'当前任务只允许prefix：保持省份拼音顺序，不跳省凑数\')\n        self.strategy = strategy\n        super().__init__(*args, **kwargs)\n\n    def finish_pending(self, file):\n        if not file or not Path(file).is_file():\n            raise RuntimeError(\'下载尚未校验；保留pending，不重复提交\')\n        pending = self.state[\'pending\']\n        self.state[\'completed\'].append({**pending, \'file\': str(file)})\n        selected = {i[\'region\'] for i in pending[\'items\']}\n        self.state[\'queue\'] = [i for i in self.state[\'queue\'] if i[\'region\'] not in selected]\n        self.state[\'pending\'] = None\n        if pending.get(\'regions\'):\n            self.state[\'province_batch_stop_day\'] = pending[\'quota_day\']\n        self.save()\n\n    def quota(self):\n        day = self.today().isoformat()\n        # 全程开始只读取一次网站余额；跨午夜不自动继续，交给下次调用。\n        self.backend.balance_day = self.state[\'current_date\']\n        balance = self.backend.balance()\n        if type(balance) is not int or not 0 <= balance <= 800:\n            raise RuntimeError(\'网站实际余额无法确认\')\n        if day != self.today().isoformat():\n            raise RuntimeError(\'读取余额时跨午夜，请重新运行\')\n        used = sum(p[\'count\'] for p in self.state[\'completed\'] if p.get(\'quota_day\') == day)\n        remaining = min(balance, max(0, self.daily_limit-used))\n        if self.state.get(\'quota_day\') == day and self.state.get(\'daily_limit\', 100) == self.daily_limit:\n            remaining = min(remaining, self.state[\'remaining\'])\n        self.state.update(quota_day=day, remaining=remaining, strategy=self.strategy, daily_limit=self.daily_limit)\n        self.save()\n        print(f\'网站余额{balance}；本程序今日剩余预算{remaining}；策略{self.strategy}\', flush=True)\n        return day\n\n    def initialize_queue(self, remaining):\n        day = self.state[\'current_date\']\n        count = self.count([])\n        if count <= remaining:\n            self.state[\'queue\'] = [{\'region\': None, \'count\': count}]\n        else:\n            names = sorted(self.backend.regions(day), key=province_key)\n            if not names or len(names) != len(set(names)):\n                raise RuntimeError(\'省份列表为空或重复\')\n            combined = self.count(names)\n            print(f\'覆盖校验：{day} 全国{count}条，全部省份组合{combined}条\', flush=True)\n            # 完整选择当天可用地区后，允许按用户授权省略非省份覆盖差额。\n            # 再查全国，避免将查询期间的数据变化误记为可省略差额。\n            if combined > count:\n                raise RuntimeError(\'省份组合比全国更多，未提交，请检查数据或筛选条件\')\n            if combined < count:\n                national_after = self.count([])\n                if national_after != count:\n                    raise RuntimeError(\'全国数量发生变化，未提交，请重新运行\')\n                omitted = {\'date\': day, \'count\': count - combined,\n                           \'national_count\': count, \'province_count\': combined,\n                           \'reason\': \'non_province_difference_user_authorized\'}\n                records = self.state.setdefault(\'omitted_national\', [])\n                records[:] = [r for r in records if r[\'date\'] != day]\n                records.append(omitted)\n                print(f\'{day} 按省份拆分：记录省略差额{count - combined}条（用户允许省略全国归属记录）\', flush=True)\n            self.state[\'national_count\'] = count\n            self.state[\'queue\'] = [{\'region\': n, \'count\': None} for n in names]\n        self.save()\n\n    def plan(self, capacity):\n        queue = self.state[\'queue\']\n        if queue[0][\'region\'] is None:\n            if queue[0][\'count\'] > capacity:\n                # 预演后降低上限，或隔天余额减少时，重新拆省，不能继续提交旧全国批次。\n                self.initialize_queue(capacity)\n                return self.plan(capacity)\n            return queue[:], queue[0][\'count\']\n        # 单调的省份前缀查询：约log2(34)次，不逐省取得数量。\n        lo, hi, total = 0, len(queue), 0\n        cache = {}\n        while lo < hi:\n            mid = (lo+hi+1)//2\n            number = self.count([i[\'region\'] for i in queue[:mid]])\n            cache[mid] = number\n            print(f\'前{mid}省：{number}条\', flush=True)\n            if number <= capacity:\n                lo, total = mid, number\n            else:\n                hi = mid-1\n        if lo:\n            total = cache[lo]\n        return queue[:lo], total\n\n    def _run(self):\n        if self.state.get(\'pending\'):\n            self.finish_pending(self.backend.recover(self.state[\'pending\']))\n        self.notify_pending()\n        if not self.next_export_day():\n            return self.stop(\'complete_with_skips\' if self.state[\'skipped\'] else \'complete\')\n        run_day = self.quota()\n        while date.fromisoformat(self.state[\'current_date\']) <= self.end:\n            if self.today().isoformat() != run_day:\n                return self.stop(\'day_changed\')\n            remaining = self.state[\'remaining\']\n            if not remaining:\n                return self.stop(\'waiting_quota\')\n            if self.state[\'queue\'] is None:\n                previous = self.backend.existing_day(self.state[\'current_date\'])\n                if previous:\n                    self.state[\'completed\'].append({\'date\': self.state[\'current_date\'], \'file\': previous[\'file\'],\n                                                   \'count\': previous[\'count\'], \'source\': \'legacy_one_day\'})\n                    self.state[\'queue\'] = []\n                else:\n                    self.initialize_queue(remaining)\n            if not self.state[\'queue\']:\n                self.state.update(current_date=(date.fromisoformat(self.state[\'current_date\'])+timedelta(days=1)).isoformat(),\n                                  queue=None, coverage_checked=False)\n                self.save()\n                continue\n            selected, count = self.plan(remaining)\n            if count == 0 and selected:\n                # 零条前缀可以直接前进，不创建订单。\n                done = {i[\'region\'] for i in selected}\n                self.state[\'queue\'] = [i for i in self.state[\'queue\'] if i[\'region\'] not in done]\n                self.save()\n                continue\n            if not selected:\n                head = self.state[\'queue\'][0]\n                fresh = self.count([head[\'region\']])\n                if fresh > 800:\n                    if not self.confirm:\n                        return self.stop(\'preview_skip\')\n                    record = {\'date\': self.state[\'current_date\'], \'region\': head[\'region\'],\n                              \'count\': fresh, \'reason\': \'province_over_daily_limit\', \'limit\': 800}\n                    self.state[\'skipped\'].append(record)\n                    self.state[\'notifications\'].append({**record, \'status\':\'pending\', \'attempts\':0})\n                    self.state[\'queue\'].pop(0)\n                    self.save()\n                    continue\n                zeros = [i for i in self.state[\'queue\'] if i.get(\'count\') == 0]\n                if zeros:\n                    self.state[\'queue\'] = [i for i in self.state[\'queue\'] if i not in zeros]\n                    self.save()\n                    continue\n                # 不把高于100条的省误认为高于网站800条；留待次日或更细拆分。\n                return self.stop(\'waiting_quota\' if remaining < self.daily_limit else \'needs_finer_split\')\n            regions = [i[\'region\'] for i in selected if i[\'region\'] is not None]\n            fresh = self.count(regions)\n            if fresh != count:\n                self.save()\n                raise RuntimeError(\'组合实际条数发生变化，未提交；请核对缓存后重新规划\')\n            balance, actual = self.backend.prepare(self.state[\'current_date\'], regions, count)\n            check = self.backend.last_order_check\n            if not isinstance(check, dict):\n                raise RuntimeError(\'缺少最终订单额度记录，禁止提交\')\n            self.state.setdefault(\'order_checks\', []).append(check)\n            self.save()\n            if type(balance) is not int or not 0 <= balance <= 800 or actual != count:\n                raise RuntimeError(\'结算条数或余额不合法\')\n            if self.today().isoformat() != run_day:\n                return self.stop(\'day_changed\')\n            # 网站最后显示更小的余额时，缩小本地预算并重新选择，绝不直接扣除。\n            if balance < remaining:\n                self.state[\'remaining\'] = balance\n                if self.state[\'queue\'][0][\'region\'] is None:\n                    self.state[\'queue\'] = None\n                self.save()\n                continue\n            if count > min(remaining, self.daily_limit):\n                raise RuntimeError(\'超过本次配置上限，未提交\')\n            print(f\'准备导出{self.state["current_date"]}：{regions or "全国"}，{count}条\', flush=True)\n            if not self.confirm:\n                return self.stop(\'preview\')\n            pending = {\'batch_id\': uuid4().hex, \'date\': self.state[\'current_date\'], \'regions\': regions,\n                       \'items\': selected, \'count\': count, \'quota_day\': run_day,\n                       \'order_check\': check}\n            self.state.update(pending=pending, remaining=remaining-count, status=\'submitting\')\n            self.save()\n            self.finish_pending(self.backend.submit(pending[\'batch_id\']))\n        return self.stop(\'complete_with_skips\' if self.state[\'skipped\'] else \'complete\')\n\n\ndef export_fast(driver, base=r\'D:\\桌面\\data\', start=\'2025-01-03\', end=None,\n                confirm=False, daily_limit=400, strategy=\'prefix\'):\n    from selenium_range import SeleniumBackend\n    from automatic_ui import DeferredNotifier\n    backend = SeleniumBackend(driver, base)\n    backend.balance_day = start\n    return FastRange(backend, base, start=start, end=end, confirm=confirm,\n                     daily_limit=daily_limit, strategy=strategy, notifier=DeferredNotifier()).run()\n', 'browser_session': '"""恢复指定 Chrome 配置目录的调试连接；不结束进程、不删除配置或导出进度。"""\nimport json\nimport os\nimport re\nimport subprocess\nimport time\nfrom pathlib import Path\nfrom urllib.request import ProxyHandler, build_opener\n\nPERSISTENT_CHROME_TASK = \'Jianyu-Automation-Chrome\'\n\n\ndef flag(command, name):\n    match = re.search(r\'--\' + re.escape(name) + r\'(?:=|\\s+)(?:"([^"]*)"|(\\S+))\', command or \'\')\n    return (match.group(1) if match.group(1) is not None else match.group(2)) if match else None\n\n\ndef profile_ports(commands, profile):\n    target = os.path.normcase(os.path.abspath(str(profile)))\n    ports, occupied = [], False\n    for command in commands:\n        value = flag(command, \'user-data-dir\')\n        if not value or os.path.normcase(os.path.abspath(value)) != target:\n            continue\n        occupied = True\n        value = flag(command, \'remote-debugging-port\')\n        if value and value.isdigit() and 0 < int(value) < 65536:\n            ports.append(int(value))\n    return list(dict.fromkeys(ports)), occupied\n\n\ndef chrome_commands():\n    result = subprocess.run([\n        \'powershell.exe\', \'-NoProfile\', \'-NonInteractive\', \'-Command\',\n        "[Console]::OutputEncoding = [System.Text.UTF8Encoding]::new(); @(Get-CimInstance Win32_Process -Filter \\"Name=\'chrome.exe\'\\" | Select-Object -ExpandProperty CommandLine) | ConvertTo-Json -Compress"\n    ], capture_output=True, text=True, encoding=\'utf-8\', errors=\'replace\', timeout=15,\n       creationflags=getattr(subprocess, \'CREATE_NO_WINDOW\', 0))\n    if result.returncode:\n        raise RuntimeError(\'无法读取 Chrome 进程以安全重连，请关闭旧自动化 Chrome 后重试。\')\n    data = json.loads(result.stdout.strip() or \'[]\')\n    return [data] if isinstance(data, str) else [v for v in (data or []) if v]\n\n\ndef endpoint(port):\n    if not 0 < int(port) < 65536:\n        return False\n    try:\n        # 本机调试连接不经过系统代理，避免 localhost 被代理导致超时。\n        with build_opener(ProxyHandler({})).open(f\'http://127.0.0.1:{port}/json/version\', timeout=1) as response:\n            info = json.load(response)\n        return bool(info.get(\'webSocketDebuggerUrl\') and \'Chrome/\' in info.get(\'Browser\', \'\'))\n    except (OSError, ValueError):\n        return False\n\n\ndef active_port(profile):\n    try:\n        value = int((Path(profile) / \'DevToolsActivePort\').read_text().splitlines()[0])\n        return value if 0 < value < 65536 else None\n    except (OSError, ValueError, IndexError):\n        return None\n\n\ndef connect_browser(chrome, base, existing=None):\n    from selenium import webdriver\n    if existing is not None:\n        try:\n            if existing.window_handles:\n                print(\'复用当前 driver。\')\n                return existing\n        except Exception:\n            pass\n    base = Path(base)\n    downloads = base / \'downloads\'\n    downloads.mkdir(parents=True, exist_ok=True)\n    commands = chrome_commands()\n    selected = None\n    profiles = [base / \'notebook_chrome\', base / \'notebook_chrome_v7\']\n    occupancy = {}\n    for profile in profiles:\n        ports, occupied = profile_ports(commands, profile)\n        occupancy[profile] = occupied\n        stored = active_port(profile)\n        if occupied and stored:\n            ports.append(stored)\n        for port in ports:\n            if endpoint(port):\n                selected = (profile, port)\n                break\n        if selected:\n            break\n    if selected:\n        profile, port = selected\n        print(\'已找到旧自动化 Chrome，重新连接：\', profile)\n    else:\n        profile = next((p for p in profiles if not occupancy[p]), None)\n        if profile is None:\n            raise RuntimeError(\'两个自动化配置目录均被占用，且调试连接不可用。请保存工作并关闭旧自动化 Chrome 窗口后重试；程序不会强制关闭浏览器。\')\n        profile.mkdir(parents=True, exist_ok=True)\n        print(\'启动常驻 Chrome：\', profile, \'（独立于本次导出任务）\')\n        process = None\n        if profile == profiles[0]:\n            started = subprocess.run(\n                [\'schtasks.exe\', \'/Run\', \'/TN\', PERSISTENT_CHROME_TASK],\n                capture_output=True, creationflags=getattr(subprocess, \'CREATE_NO_WINDOW\', 0)).returncode == 0\n        else:\n            started = False\n        if not started:\n            process = subprocess.Popen([\n                str(chrome), \'--remote-debugging-port=0\', \'--remote-debugging-address=127.0.0.1\',\n                f\'--user-data-dir={profile.resolve()}\', \'--no-first-run\', \'--no-default-browser-check\',\n                \'https://www.jianyu360.cn/\'\n            ], stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,\n               creationflags=(getattr(subprocess, \'DETACHED_PROCESS\', 0)\n                              | getattr(subprocess, \'CREATE_NEW_PROCESS_GROUP\', 0)))\n        deadline = time.monotonic() + 40\n        while time.monotonic() < deadline:\n            port = active_port(profile)\n            if port and endpoint(port):\n                break\n            if process is not None and process.poll() is not None:\n                raise RuntimeError(\'Chrome 启动进程已退出，可能被已有窗口接管或被系统阻止。未执行导出。\')\n            time.sleep(0.3)\n        else:\n            raise RuntimeError(\'40秒内未取得 Chrome 调试服务。请检查 Chrome 是否正常打开、是否有系统拦截；未执行导出。\')\n    options = webdriver.ChromeOptions()\n    options.binary_location = str(chrome)\n    options.debugger_address = f\'127.0.0.1:{port}\'\n    driver = webdriver.Chrome(options=options)\n    driver.execute_cdp_cmd(\'Browser.setDownloadBehavior\', {\n        \'behavior\': \'allow\', \'downloadPath\': str(downloads.resolve())})\n    print(\'已连接 Chrome。请确认登录后运行预演。\')\n    return driver\n', 'repair_start': '"""仅供用户主动执行的范围调整：备份并保留扣额与完成记录，不操作网站。"""\nimport json\nfrom datetime import date, datetime\nfrom pathlib import Path\nfrom uuid import uuid4\n\n\ndef repair_start(base, requested=\'2025-01-03\'):\n    requested = date.fromisoformat(requested).isoformat()\n    base = Path(base)\n    path = base / \'range_progress.json\'\n    lock = base / \'one_day.lock\'\n    with lock.open(\'x\', encoding=\'utf-8\') as f:\n        f.write(\'调整任务范围中\')\n    try:\n        original = path.read_bytes()\n        state = json.loads(original)\n        if state.get(\'pending\'):\n            raise RuntimeError(\'存在待恢复订单，请先恢复原订单，不调整日期\')\n        current = state[\'current_date\']\n        if requested < state[\'start\']:\n            raise RuntimeError(\'不允许向前扩大范围\')\n        if current < requested and (state.get(\'queue\') not in (None, []) or\n                any(r.get(\'date\') == current for r in state.get(\'completed\', []))):\n            raise RuntimeError(\'当前日期存在省份批次，不能跳过未完成范围\')\n        if state[\'start\'] == requested and current >= requested:\n            print(\'范围已经正确，无需修改\')\n            return\n        backup = path.with_name(\'range_progress.before_start_fix_\' + uuid4().hex + \'.json\')\n        backup.write_bytes(original)\n        state.setdefault(\'range_adjustments\', []).append({\n            \'old_start\': state[\'start\'], \'old_current_date\': current,\n            \'requested_start\': requested, \'at\': datetime.now().isoformat()})\n        state[\'start\'] = requested\n        if current < requested:\n            state.update(current_date=requested, queue=None, coverage_checked=False)\n        state[\'status\'] = \'ready\'\n        temp = path.with_suffix(\'.tmp\')\n        temp.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n        temp.replace(path)\n        print(\'已调整：\', requested, \'当前日期：\', state[\'current_date\'])\n        print(\'保留原完成记录、每日已用额度和下载文件；备份：\', backup)\n    finally:\n        lock.unlink(missing_ok=True)\n', 'auto_login': '"""只填写剑鱼站点可见且唯一的密码登录表单；验证码出现时停止。"""\nfrom urllib.parse import urlparse\n\n\nclass PasswordLogin:\n    def __init__(self, username, password, base):\n        self.username, self.password, self.base = username, password, base\n\n    def ensure(self, driver):\n        from selenium.webdriver.common.by import By\n        from selenium.webdriver.support.ui import WebDriverWait\n        from selenium.common.exceptions import TimeoutException\n        from export_one_day import OneDay, FILTER_URL\n        driver.switch_to.default_content()\n        driver.get(FILTER_URL)\n        job = OneDay(driver, \'2025-01-03\', self.base, False)\n\n        def visible(by, selector):\n            return [e for e in driver.find_elements(by, selector) if e.is_displayed()]\n\n        def logged_in():\n            text = job.body()\n            return (\'筛选日期\' in text and \'关键词匹配方式\' in text\n                    and not password_fields() and not login_switches())\n\n        def password_fields():\n            return visible(By.CSS_SELECTOR, \'input[placeholder="请输入手机号或账号名"]\')\n\n        def login_switches():\n            return visible(By.XPATH, "//*[self::a or self::span or self::div or self::button][normalize-space(.)=\'验证码/密码登录\' or normalize-space(.)=\'密码登录\'][not(.//*[normalize-space(.)=\'密码登录\' or normalize-space(.)=\'验证码/密码登录\'])]")\n\n        try:\n            job.find_page(lambda: logged_in() or bool(password_fields()) or bool(login_switches()), timeout=30)\n        except TimeoutException:\n            raise RuntimeError(\'未识别到筛选页或密码登录入口；需要检查登录页面，未执行导出。\') from None\n        if logged_in():\n            print(\'登录状态有效。\', flush=True)\n            return\n        if not self.username or not self.password:\n            raise RuntimeError(\'登录已失效且未配置账号密码。请登录后重新启动调度。\')\n        if not password_fields():\n            switches = login_switches()\n            if len(switches) != 1:\n                raise RuntimeError(\'密码登录入口不唯一，未填写密码。\')\n            switches[0].click()\n        WebDriverWait(driver, 10).until(lambda _: len(password_fields()) == 1)\n        # 不向被重定向的其他域名或第三方iframe填写凭据。\n        host = urlparse(driver.execute_script(\'return location.href\')).hostname or \'\'\n        if host != \'jianyu360.cn\' and not host.endswith(\'.jianyu360.cn\'):\n            raise RuntimeError(\'当前表单不是剑鱼域名，未填写凭据。\')\n        usernames = password_fields()\n        passwords = visible(By.CSS_SELECTOR, \'input[type="password"][placeholder="输入密码"]\')\n        if len(usernames) != 1 or len(passwords) != 1:\n            raise RuntimeError(\'账号密码表单不唯一，未填写。\')\n        user, secret = usernames[0], passwords[0]\n        scope = secret\n        buttons = []\n        for _ in range(8):\n            scope = scope.find_element(By.XPATH, \'..\')\n            if not scope.find_elements(By.CSS_SELECTOR, \'input[placeholder="请输入手机号或账号名"]\'):\n                continue\n            buttons = [e for e in scope.find_elements(By.XPATH,\n                       ".//*[self::button or self::a or self::input or self::div][translate(normalize-space(.),\' \',\'\')=\'登录\' or @value=\'登录\'][not(.//*[translate(normalize-space(.),\' \',\'\')=\'登录\'])]")\n                       if e.is_displayed()]\n            if buttons:\n                break\n        if len(buttons) != 1:\n            raise RuntimeError(\'未唯一识别账号密码表单的登录按钮，未提交凭据。\')\n        user.clear()\n        user.send_keys(self.username)\n        secret.clear()\n        secret.send_keys(self.password)\n        WebDriverWait(driver, 10).until(lambda _: buttons[0].is_enabled())\n        buttons[0].click()\n        try:\n            WebDriverWait(driver, 30).until(lambda _: not password_fields())\n            driver.switch_to.default_content()\n            driver.get(FILTER_URL)\n            job.find_page(logged_in, timeout=30)\n        except TimeoutException:\n            raise RuntimeError(\'自动登录未完成：可能需要验证码、滑块、扫码或账号密码有误。已停止，不自动反复尝试密码。\') from None\n        print(\'账号密码登录成功。\', flush=True)\n', 'daily_runner': '"""Notebook 内持续调度；北京时间次日03:00续跑，不创建订单重试循环。"""\nimport time\nfrom datetime import datetime, timedelta, timezone\nfrom pathlib import Path\n\nCHINA = timezone(timedelta(hours=8))\n\n\ndef china_now():\n    return datetime.now(CHINA)\n\n\ndef next_run(now):\n    return (now.astimezone(CHINA) + timedelta(days=1)).replace(hour=3, minute=0, second=0, microsecond=0)\n\n\ndef run_daily(export, get_driver, before_run, base, start=\'2025-01-03\', end=None,\n              daily_limit=400, now=china_now, sleep=time.sleep, first_run_at=None):\n    # 防止长时间等待期间再开启第二个调度器；与批次运行锁分离。\n    lock = Path(base) / \'daily_scheduler.lock\'\n    lock.parent.mkdir(parents=True, exist_ok=True)\n    try:\n        with lock.open(\'x\', encoding=\'utf-8\') as f:\n            f.write(\'自动调度中；仅在确认旧内核已停止后清除此锁\')\n    except FileExistsError:\n        raise RuntimeError(\'已有自动调度器运行或遗留锁，请先确认旧Notebook已停止。\')\n    try:\n        due = first_run_at if first_run_at is not None else next_run(now())\n        if due.tzinfo is None:\n            raise ValueError(\'首次运行时间必须带时区\')\n        print(\'已就绪；首次自动运行：\', due.astimezone(CHINA).isoformat(), flush=True)\n        while now() < due:\n            sleep(min(30, max(0.1, (due - now()).total_seconds())))\n        while True:\n            driver = get_driver()\n            before_run(driver)\n            print(f\'[{now().isoformat()}] 开始续跑，每日最多{daily_limit}条\', flush=True)\n            state = export(driver, base=base, start=start, end=end,\n                           confirm=True, daily_limit=daily_limit, strategy=\'prefix\')\n            status = state[\'status\']\n            print(\'本轮状态：\', status, \'日期：\', state[\'current_date\'], \'剩余预算：\', state[\'remaining\'], flush=True)\n            if status not in {\'waiting_quota\', \'province_batch_done\', \'day_changed\', \'complete\', \'complete_with_skips\'}:\n                print(\'需要处理后才能继续，调度停止：\', status, flush=True)\n                return state\n            if end is not None and status in {\'complete\', \'complete_with_skips\'}:\n                return state\n            # 无固定截止日期时，完成当前范围后明天继续处理新增日期。\n            due = next_run(now())\n            if status == \'day_changed\':\n                due = now().astimezone(CHINA).replace(hour=3, minute=0, second=0, microsecond=0)\n                if due <= now():\n                    continue\n            print(\'保持本单元格运行；下次自动运行：\', due.isoformat(), flush=True)\n            while now() < due:\n                sleep(min(30, max(0.1, (due - now()).total_seconds())))\n    finally:\n        lock.unlink(missing_ok=True)\n'}
# 计划任务优先读取同目录新版脚本；Notebook内嵌代码仅作离线后备。
for _module_name in list(MODULE_SOURCES):
    _source_file = Path.cwd() / (_module_name + ".py")
    if _source_file.is_file():
        MODULE_SOURCES[_module_name] = _source_file.read_text(encoding="utf-8")

for name, source in MODULE_SOURCES.items():
    module = types.ModuleType(name)
    module.__file__ = str(Path.cwd() / ("embedded_" + name + ".py"))
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
run_export = sys.modules["fast_range"].export_fast
print("完整程序已加载；每日400条，尚未启动。")


## 3．配置及恢复进度
保留原目录，不删除已完成记录。每日预算按400减当天已完成条数计算，再与网站实际余额取较小值。

In [ ]:
import json
from pathlib import Path
BASE_DIR = r"D:\桌面\data"
START_DATE = "2025-01-03"
END_DATE = None
DAILY_LIMIT = 400
progress = Path(BASE_DIR) / "range_progress.json"
if progress.is_file():
    saved = json.loads(progress.read_text(encoding="utf-8"))
    if saved["start"] != START_DATE or saved["current_date"] < START_DATE:
        sys.modules["repair_start"].repair_start(BASE_DIR, START_DATE)
        saved = json.loads(progress.read_text(encoding="utf-8"))
    print("开始日期：", saved["start"], "继续日期：", saved["current_date"])
    print("原剩余预算：", saved.get("remaining"), "待恢复订单：", bool(saved.get("pending")))
else:
    print("未找到旧进度，将从", START_DATE, "开始。请确认原进度目录正确：", BASE_DIR)

from datetime import datetime
FIRST_RUN_AT = datetime(2026, 9, 14, 19, 45, tzinfo=sys.modules["daily_runner"].CHINA)
print("首次运行：", FIRST_RUN_AT.isoformat())


## 4．填写自动登录账号和密码
仅需本次启动输入一次。密码隐藏输入，不要填写在代码中。已经登录且不需要自动重登时可留空，但登录失效后程序将停止。

In [ ]:
from getpass import getpass
account = input("剑鱼手机号或账号名：").strip()
password = getpass("剑鱼登录密码（隐藏输入）：") if account else ""
login = sys.modules["auto_login"].PasswordLogin(account, password, BASE_DIR)
del password
print("登录配置完成，密码未写入文件。")


## 5．连接浏览器
复用有效driver或重连原自动化Chrome。

In [ ]:
import os
import subprocess
from pathlib import Path

URL = (
    "https://www.jianyu360.cn/page_workDesktop/work-bench/page"
    "?link=https%3A%2F%2Fwww.jianyu360.cn"
    "%2Ffront%2FdataExport%2FtoSieve"
)


def find_chrome():
    # 查找 Windows 注册表中的 Chrome 安装路径
    import winreg

    registry_path = (
        r"SOFTWARE\Microsoft\Windows"
        r"\CurrentVersion\App Paths\chrome.exe"
    )

    for root in (winreg.HKEY_CURRENT_USER, winreg.HKEY_LOCAL_MACHINE):
        for view in (winreg.KEY_WOW64_64KEY, winreg.KEY_WOW64_32KEY):
            try:
                with winreg.OpenKey(
                    root, registry_path, 0, winreg.KEY_READ | view
                ) as key:
                    value, _ = winreg.QueryValueEx(key, "")
                    path = Path(os.path.expandvars(value.strip('"')))
                    if path.is_file():
                        return path
            except OSError:
                continue

    # 查找常见安装目录
    for variable in ("PROGRAMFILES", "PROGRAMFILES(X86)", "LOCALAPPDATA"):
        base = os.environ.get(variable)
        if base:
            path = Path(base) / "Google/Chrome/Application/chrome.exe"
            if path.is_file():
                return path

    raise FileNotFoundError("未找到谷歌浏览器，请确认已安装 Google Chrome。")



driver = sys.modules["browser_session"].connect_browser(
    find_chrome(), BASE_DIR, existing=globals().get("driver")
)
print("浏览器窗口数：", len(driver.window_handles))


## 6．启动每日自动导出
运行后等待首次指定时间，到时正式导出，不再询问。此单元格保持[*]运行是正常现象，每天03:00自动续跑；停止请使用Jupyter“中断内核”。不要同时启动另一份Notebook。

In [ ]:
import sys

source = MODULE_SOURCES["fast_range"]

# 取消“当天完成过一批省份就不能继续”
source = source.replace(
    "        if (self.state.get('province_batch_stop_day') == self.today().isoformat()\n"
    "                and self.state.get('daily_limit', 100) == self.daily_limit):\n"
    "            return self.stop('province_batch_done')\n",
    ""
)

# 取消“刚完成一批省份就停止”
source = source.replace(
    "            if regions:\n"
    "                return self.stop('province_batch_done')\n",
    ""
)

assert "return self.stop('province_batch_done')" not in source, \
    "当前版本不匹配，请把提示发给我。"

MODULE_SOURCES["fast_range"] = source
module = sys.modules["fast_range"]
exec(compile(source, module.__file__, "exec"), module.__dict__)
run_export = module.export_fast

print("修正完成：连续处理后续日期，每天累计最多400条。")

In [ ]:
def current_driver():
    global driver
    driver = sys.modules["browser_session"].connect_browser(
        find_chrome(), BASE_DIR, existing=globals().get("driver"))
    return driver

try:
    final_state = sys.modules["daily_runner"].run_daily(
        run_export, current_driver, login.ensure,
        base=BASE_DIR, start=START_DATE, end=END_DATE, daily_limit=400, first_run_at=FIRST_RUN_AT,
    )
except KeyboardInterrupt:
    print("已停止自动调度。已保存进度；若中断发生在提交中，下次仅恢复原订单。")


In [ ]:
from selenium_range import SeleniumBackend
from export_range import province_key

check = SeleniumBackend(driver, BASE_DIR)
day = "2025-01-06"

national_before = check.query(day, [])
regions = sorted(check.regions(day), key=province_key)
combined = check.query(day, regions)
national_after = check.query(day, [])

print("第一次全国：", national_before)
print("全部省份组合：", combined)
print("第二次全国：", national_after)
print("省份数量：", len(regions))
print("省份名单：", regions)

## 需要知道的停止情况
省份数量校验失败、下载结果不确定、登录额外验证或单省超过400条不能拆分时，程序停止而不是重复扣额。
等待次日时会显示下次运行时间。若电脑睡眠，程序无法准点运行，恢复后会继续；内核关闭后需重新运行Notebook并输入密码。
下载记录继续保存于 D:\桌面\data。原状态的100条上限会切换为400条，但仍扣除当天已经导出的数量。
本地调度测试验证了次日03:00自动调用、confirm=True、400上限传递及异常不重复提交；实际登录控件和整夜运行尚未实测。
